# RNN によるテキスト生成

### 学習目標

- 文字ベースの RNN を使用してテキストを生成する方法を学ぶ
- テキスト生成用のトレーニング サンプルとターゲットを作成する
- Keras のサブクラス化を使用してシーケンス生成用の RNN モデルを構築する
- テキスト ジェネレーターを作成し、出力を評価する

このチュートリアルでは、文字ベースの RNN を使用してテキストを生成する方法を説明します。Andrej Karpathy の [The Unreasonable Effectiveness of Recurrent Neural Networks](http://karpathy.github.io/2015/05/21/rnn-effectiveness/) にあるシェイクスピアの著作のデータセットを使用します。このデータからの文字のシーケンス（"Shakespear"）が与えられたとき、シーケンスの次の文字（"e"）を予測するようにモデルをトレーニングします。モデルを繰り返し呼び出すことで、より長いテキスト シーケンスを生成できます。

以下は、このチュートリアルのモデルを 30 エポックトレーニングし、プロンプト「Q」で開始したときのサンプル出力です。

<pre>
QUEENE:
I had thought thou hadst a Roman; for the oracle,
Thus by All bids the man against the word,
Which are so weak of care, by old care done;
Your children were in your holy love,
And the precipitation through the bleeding throne.

BISHOP OF ELY:
Marry, and will, my lord, to weep in such a one were prettiest;
Yet now I was adopted heir
Of the world's lamentable day,
To watch the next way with his father with his face?

ESCALUS:
The cause why then we are all resolved more sons.

VOLUMNIA:
O, no, no, no, no, no, no, no, no, no, no, no, no, no, no, no, no, no, no, no, no, it is no sin it should be dead,
And love and pale as any will to that word.

QUEEN ELIZABETH:
But how long have I heard the soul for this world,
And show his hands of life be proved to stand.

PETRUCHIO:
I say he look'd on, if I must be content
To stay him from the fatal of our country's bliss.
His lordship pluck'd from this sentence then for prey,
And then let us twain, being the moon,
were she such a case as fills m
</pre>

一部の文は文法的に正しいものの、ほとんどは意味をなしません。モデルは単語の意味を学習していませんが、以下の点に注目してください。

* モデルは文字ベースです。トレーニング開始時、モデルは英語の単語の綴り方や、単語がテキストの単位であることさえ知りませんでした。

* 出力の構造は戯曲に似ています。テキストのブロックは通常、データセットと同様にすべて大文字の登場人物名から始まります。

* 以下で示すように、モデルはテキストの小さなバッチ（各 100 文字）でトレーニングされていますが、一貫した構造を持つより長いテキスト シーケンスを生成することができます。

## セットアップ

### TensorFlow とその他のライブラリのインポート

In [ ]:
import os
import warnings

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [ ]:
import time

import numpy as np
import tensorflow as tf

### シェイクスピア データセットのダウンロード

独自のデータでこのコードを実行するには、次の行を変更してください。

In [ ]:
path_to_file = tf.keras.utils.get_file(
    "shakespeare.txt",
    "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt",
)

### データの読み込み

まずファイルをダウンロードし、次にデコードします。

In [ ]:
text = open(path_to_file, "rb").read().decode(encoding="utf-8")
print(f"Length of text: {len(text)} characters")

テキストの最初の 250 文字を見てみましょう。

In [ ]:
print(text[:250])

コーパス/ドキュメント内にいくつのユニークな文字があるか確認しましょう。

In [ ]:
vocab = sorted(set(text))
print(f"{len(vocab)} unique characters")

## テキストの処理

### テキストのベクトル化

トレーニングの前に、文字列を数値表現に変換する必要があります。

#### TODO 1
`tf.keras.layers.StringLookup` レイヤーを使用すると、各文字を数値 ID に変換できます。事前にテキストをトークンに分割するだけで十分です。

In [ ]:
example_texts = ["abcdefg", "xyz"]

# TODO 1 
chars = # ここにコードを挿入

`tf.keras.layers.StringLookup` レイヤーを作成します。

In [ ]:
ids_from_chars = tf.keras.layers.StringLookup(
    vocabulary=list(vocab), mask_token=None
)

トークンから文字 ID に変換されます。

In [ ]:
ids = ids_from_chars(chars)
ids

このチュートリアルの目的はテキストを生成することなので、この表現を反転して人間が読める文字列を復元することも重要になります。これには `tf.keras.layers.StringLookup(..., invert=True)` を使用できます。

注: ここでは、`sorted(set(text))` で生成された元の語彙を渡す代わりに、`tf.keras.layers.StringLookup` レイヤーの `get_vocabulary()` メソッドを使用して、`[UNK]` トークンが同じように設定されるようにします。

In [ ]:
chars_from_ids = tf.keras.layers.StringLookup(
    vocabulary=ids_from_chars.get_vocabulary(), invert=True, mask_token=None
)

このレイヤーは ID のベクトルから文字を復元し、文字の `tf.RaggedTensor` として返します。

In [ ]:
chars = chars_from_ids(ids)
chars

`tf.strings.reduce_join` を使用して、文字を結合して文字列に戻すことができます。

In [ ]:
tf.strings.reduce_join(chars, axis=-1).numpy()

In [ ]:
def text_from_ids(ids):
    return tf.strings.reduce_join(chars_from_ids(ids), axis=-1)

### 予測タスク

ある文字、または文字のシーケンスが与えられたとき、最も可能性の高い次の文字は何でしょうか？これがモデルをトレーニングして実行させるタスクです。モデルへの入力は文字のシーケンスであり、各タイムステップで次の文字となる出力を予測するようにモデルをトレーニングします。

RNN は以前に現れた要素に依存する内部状態を維持するため、現時点までに計算されたすべての文字が与えられたとき、次の文字は何になるでしょうか？

### トレーニング サンプルとターゲットの作成

次に、テキストをサンプル シーケンスに分割します。各入力シーケンスには、テキストからの `seq_length` 個の文字が含まれます。

各入力シーケンスについて、対応するターゲットには同じ長さのテキストが含まれますが、1 文字分右にシフトされています。

したがって、テキストを `seq_length+1` のチャンクに分割します。たとえば、`seq_length` が 4 でテキストが「Hello」の場合、入力シーケンスは「Hell」、ターゲット シーケンスは「ello」になります。

#### TODO 2
まず `tf.data.Dataset.from_tensor_slices` 関数を使用して、テキスト ベクトルを文字インデックスのストリームに変換します。

In [ ]:
# TODO 2
all_ids = # ここにコードを挿入
all_ids

In [ ]:
ids_dataset = tf.data.Dataset.from_tensor_slices(all_ids)

In [ ]:
for ids in ids_dataset.take(10):
    print(chars_from_ids(ids).numpy().decode("utf-8"))

In [ ]:
seq_length = 100
examples_per_epoch = len(text) // (seq_length + 1)

`batch` メソッドを使用すると、これらの個々の文字を目的のサイズのシーケンスに簡単に変換できます。

In [ ]:
sequences = ids_dataset.batch(seq_length + 1, drop_remainder=True)

for seq in sequences.take(1):
    print(chars_from_ids(seq))

トークンを結合して文字列に戻すと、何が行われているかがわかりやすくなります。

In [ ]:
for seq in sequences.take(5):
    print(text_from_ids(seq).numpy())

トレーニングには、`(input, label)` ペアのデータセットが必要になります。ここで、`input` と `label` はシーケンスです。各タイムステップにおいて、入力は現在の文字であり、ラベルは次の文字です。

以下は、シーケンスを入力として受け取り、複製してシフトすることで、各タイムステップの入力とラベルを揃える関数です。

In [ ]:
def split_input_target(sequence):
    input_text = sequence[:-1]
    target_text = sequence[1:]
    return input_text, target_text

In [ ]:
split_input_target(list("Tensorflow"))

In [ ]:
dataset = sequences.map(split_input_target)

In [ ]:
for input_example, target_example in dataset.take(1):
    print("Input :", text_from_ids(input_example).numpy())
    print("Target:", text_from_ids(target_example).numpy())

### トレーニング バッチの作成

`tf.data` を使用して、テキストを扱いやすいシーケンスに分割しました。しかし、このデータをモデルに入力する前に、データをシャッフルしてバッチにパックする必要があります。

In [ ]:
# バッチサイズ
BATCH_SIZE = 64

# データセットをシャッフルするためのバッファサイズ
# （tf.data は無限のシーケンスでも動作するように設計されているため、
# メモリ内でシーケンス全体をシャッフルしようとはしません。代わりに、
# 要素をシャッフルするためのバッファを維持します）
BUFFER_SIZE = 10000

dataset = (
    dataset.shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.experimental.AUTOTUNE)
)

dataset

## モデルの構築

このセクションでは、モデルを `keras.Model` のサブクラスとして定義します（詳細については [サブクラス化による新しいレイヤーとモデルの作成](https://www.tensorflow.org/guide/keras/custom_layers_and_models) を参照してください）。

#### TODO 3 次のレイヤーでモデルを構築する

* `tf.keras.layers.Embedding`: 入力レイヤー。各文字 ID を `embedding_dim` 次元のベクトルにマッピングするトレーニング可能なルックアップ テーブル。
* `tf.keras.layers.GRU`: サイズ `units=rnn_units` の RNN の一種（ここでは LSTM レイヤーを使用することもできます）。
* `tf.keras.layers.Dense`: 出力レイヤーで、`vocab_size` 個の出力を持ちます。語彙内の各文字に対して 1 つのロジットを出力します。これらは、モデルによる各文字の対数尤度（log-likelihood）です。

In [ ]:
# 文字の語彙数
vocab_size = len(vocab)

# 埋め込み次元数
embedding_dim = 256

# RNN ユニット数
rnn_units = 1024

以下のクラスは次の処理を行います。
- `tf.keras.Model` からクラスを派生させます
- コンストラクタを使用してモデルのレイヤーを定義します
- コンストラクタで定義されたレイヤーを使用して順伝播を定義します

In [ ]:
class MyModel(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, rnn_units):
        super().__init__(self)
        # TODO - 埋め込みレイヤーを作成
        self.embedding = # ここにコードを挿入
        
        # TODO - GRU レイヤーを作成
        self.gru = # ここにコードを挿入
        
        # TODO - 最後に Dense レイヤーと接続
        self.dense = # ここにコードを挿入

    def call(self, inputs, states=None, return_state=False, training=False):
        x = self.embedding(inputs, training=training)
        # テキスト生成モデルをトレーニングしているため、 
        # トレーニングでは前の状態（state）を使用します。状態がない場合は、
        # 状態を初期化します
        if states is None:
            states = self.gru.get_initial_state(x)
        x, states = self.gru(x, initial_state=states, training=training)
        x = self.dense(x, training=training)

        if return_state:
            return x, states
        else:
            return x

In [ ]:
model = MyModel(
    # 語彙サイズが StringLookup レイヤーと一致していることを確認します
    vocab_size=len(ids_from_chars.get_vocabulary()),
    embedding_dim=embedding_dim,
    rnn_units=rnn_units,
)

各文字に対して、モデルは埋め込みを検索し、その埋め込みを入力として GRU を 1 タイムステップ実行し、Dense レイヤーを適用して次の文字の対数尤度を予測するロジットを生成します。

## モデルのテスト

モデルを実行して、期待どおりに動作することを確認します。

まず出力の shape を確認します。

In [ ]:
for input_example_batch, target_example_batch in dataset.take(1):
    example_batch_predictions = model(input_example_batch)
    print(
        example_batch_predictions.shape,
        "# (batch_size, sequence_length, vocab_size)",
    )

上記の例では入力のシーケンス長は `100` ですが、モデルは任意の長さの入力に対して実行できます。

In [ ]:
model.summary()

モデルから実際の予測を取得するには、出力分布からサンプリングして実際の文字インデックスを取得する必要があります。この分布は、文字の語彙に対するロジットによって定義されます。

注: 分布の _argmax_ を取るとモデルがループに陥りやすいため、この分布から _サンプリング_ することが重要です。

バッチ内の最初のサンプルで試してみましょう。

In [ ]:
sampled_indices = tf.random.categorical(
    example_batch_predictions[0], num_samples=1
)
sampled_indices = tf.squeeze(sampled_indices, axis=-1).numpy()

これにより、各タイムステップで次の文字インデックスの予測が得られます。

In [ ]:
sampled_indices

これらをデコードして、この未トレーニングのモデルによって予測されたテキストを確認します。

In [ ]:
print("Input:\n", text_from_ids(input_example_batch[0]).numpy())
print()
print("Next Char Predictions:\n", text_from_ids(sampled_indices).numpy())

## モデルのトレーニング

この時点で、問題は標準的な分類問題として扱うことができます。前の RNN 状態とこのタイムステップの入力が与えられたとき、次の文字のクラスを予測します。

#### TODO 4 オプティマイザと損失関数をアタッチする
予測の最後の次元にわたって適用されるため、このケースでは標準の `tf.keras.losses.sparse_categorical_crossentropy` 損失関数が機能します。

モデルがロジットを返すため、`from_logits` フラグを設定する必要があります。

In [ ]:
# TODO - ここに損失関数を追加
loss = # ここにコードを挿入

In [ ]:
example_batch_mean_loss = loss(target_example_batch, example_batch_predictions)
print(
    "Prediction shape: ",
    example_batch_predictions.shape,
    " # (batch_size, sequence_length, vocab_size)",
)
print("Mean loss:        ", example_batch_mean_loss)

新しく初期化されたモデルは自身の予測に過度の確信を持つべきではなく、出力ロジットはすべて同程度の大きさである必要があります。これを確認するには、平均損失の指数が語彙サイズとほぼ等しいことを確認します。損失がはるかに高い場合は、モデルが間違った回答を確信しており、初期化が不適切であることを意味します。

In [ ]:
tf.exp(example_batch_mean_loss).numpy()

`tf.keras.Model.compile` メソッドを使用してトレーニング手順を設定します。デフォルトの引数を持つ `tf.keras.optimizers.Adam` と損失関数を使用します。

In [ ]:
model.compile(optimizer="adam", loss=loss)

### チェックポイントの設定

トレーニング中にチェックポイントが確実に保存されるように、`tf.keras.callbacks.ModelCheckpoint` を使用します。

In [ ]:
# チェックポイントが保存されるディレクトリ
checkpoint_dir = "./training_checkpoints"
# チェックポイント ファイルの名前
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt_{epoch}")

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_prefix, save_weights_only=True
)

### トレーニングの実行

トレーニング時間を妥当な範囲に抑えるため、10 エポックを使用してモデルをトレーニングします。Colab では、トレーニングを高速化するためにランタイムを GPU に設定してください。

In [ ]:
EPOCHS = 10

In [ ]:
history = model.fit(dataset, epochs=EPOCHS, callbacks=[checkpoint_callback])

#### TODO 5 テキストの生成

このモデルでテキストを生成する最も簡単な方法は、ループでモデルを実行し、実行時にモデルの内部状態を追跡することです。

モデルを呼び出すたびに、テキストと内部状態を渡します。モデルは次の文字の予測とその新しい状態を返します。予測と状態をモデルに戻して渡し、テキストの生成を継続します。

以下は 1 ステップの予測を行います。

In [ ]:
class OneStep(tf.keras.Model):
    def __init__(self, model, chars_from_ids, ids_from_chars, temperature=1.0):
        super().__init__()
        self.temperature = temperature
        self.model = model
        self.chars_from_ids = chars_from_ids
        self.ids_from_chars = ids_from_chars

        # "[UNK]" が生成されるのを防ぐためのマスクを作成
        skip_ids = self.ids_from_chars(["[UNK]"])[:, None]
        sparse_mask = tf.SparseTensor(
            # 無効な各インデックスに -inf を配置
            values=[-float("inf")] * len(skip_ids),
            indices=skip_ids,
            # 語彙の shape に合わせる
            dense_shape=[len(ids_from_chars.get_vocabulary())],
        )
        self.prediction_mask = tf.sparse.to_dense(sparse_mask)
    
    # TODO 5 - 以下のコードを記述してテキストを生成
    @tf.function
    def generate_one_step(self, inputs, states=None):
        # 文字列をトークン ID に変換
        input_chars = tf.strings.unicode_split(inputs, "UTF-8")
        input_ids = self.ids_from_chars(input_chars).to_tensor()

        # モデルを実行
        # predicted_logits の shape は [batch, char, next_char_logits]
        predicted_logits, states = # ここにコードを挿入
        
        # 最後の予測のみを使用
        predicted_logits = predicted_logits[:, -1, :]
        predicted_logits = predicted_logits / self.temperature
        
        # 予測マスクを適用: "[UNK]" の生成を防止
        predicted_logits = # ここにコードを挿入

        # 出力ロジットをサンプリングしてトークン ID を生成
        predicted_ids = # ここにコードを挿入
        predicted_ids = # ここにコードを挿入

        # トークン ID から文字に変換
        predicted_chars = self.chars_from_ids(predicted_ids)

        # 文字とモデルの状態を返す
        return predicted_chars, states

In [ ]:
one_step_model = OneStep(model, chars_from_ids, ids_from_chars)

ループで実行してテキストを生成します。生成されたテキストを見ると、モデルは大文字にするタイミングや段落の作成方法を把握しており、シェイクスピア風の文体を模倣していることがわかります。トレーニング エポック数が少ないため、一貫性のある文を形成することはまだ学習できていません。

In [ ]:
start = time.time()
states = None
next_char = tf.constant(["ROMEO:"])
result = [next_char]

for n in range(1000):
    next_char, states = one_step_model.generate_one_step(
        next_char, states=states
    )
    result.append(next_char)

result = tf.strings.join(result)
end = time.time()
print(result[0].numpy().decode("utf-8"), "\n\n" + "_" * 80)
print("\nRun time:", end - start)

結果を改善するために最も簡単にできることは、より長くトレーニングすることです（`EPOCHS = 30` を試してください）。

また、別の開始文字列を試したり、モデルの精度を向上させるために別の RNN レイヤーを追加したり、温度（temperature）パラメータを調整して予測のランダム性を増減させたりすることもできます。

モデルによるテキスト生成を *より高速* に行いたい場合、最も簡単な方法はテキスト生成をバッチ化することです。以下の例では、モデルは上記で 1 つの出力を生成するのとほぼ同じ時間で 5 つの出力を生成します。

In [ ]:
start = time.time()
states = None
next_char = tf.constant(["ROMEO:", "ROMEO:", "ROMEO:", "ROMEO:", "ROMEO:"])
result = [next_char]

for n in range(1000):
    next_char, states = one_step_model.generate_one_step(
        next_char, states=states
    )
    result.append(next_char)

result = tf.strings.join(result)
end = time.time()
for s in result:
    print(s.numpy().decode("utf-8"))
print("\n\n" + "_" * 80)
print("\nRun time:", end - start)

## ジェネレーターのエクスポート

このシングルステップ モデルは簡単に [保存および復元](https://www.tensorflow.org/guide/saved_model) できるため、`tf.saved_model` がサポートされている場所であればどこでも使用できます。

In [ ]:
tf.saved_model.save(one_step_model, "one_step")
one_step_reloaded = tf.saved_model.load("one_step")

In [ ]:
states = None
next_char = tf.constant(["ROMEO:"])
result = [next_char]

for n in range(100):
    next_char, states = one_step_reloaded.generate_one_step(
        next_char, states=states
    )
    result.append(next_char)

print(tf.strings.join(result)[0].numpy().decode("utf-8"))

## 高度なトピック: カスタマイズされたトレーニング

上記の手順はシンプルですが、細かい制御はあまりできません。
teacher-forcing（教師強制）を使用しているため、誤った予測がモデルにフィードバックされるのを防ぎますが、モデルが間違いから回復することを学習しません。

モデルを手動で実行する方法を理解したので、次はトレーニング ループを実装します。これにより、たとえばモデルのオープンループ出力を安定させるために _カリキュラム学習（curriculum learning）_ を実装したい場合などの出発点が得られます。

カスタム トレーニング ループの最も重要な部分は、トレーニング ステップ関数（train step function）です。

勾配を追跡するには `tf.GradientTape` を使用します。このアプローチの詳細については、[Eager Execution ガイド](https://www.tensorflow.org/guide/eager) を参照してください。

基本的な手順は次のとおりです。

1. モデルを実行し、`tf.GradientTape` の下で損失を計算します。
2. 更新を計算し、オプティマイザを使用してモデルに適用します。

In [ ]:
class CustomTraining(MyModel):
    @tf.function
    def train_step(self, inputs):
        inputs, labels = inputs
        with tf.GradientTape() as tape:
            predictions = self(inputs, training=True)
            loss = self.loss(labels, predictions)
        grads = tape.gradient(loss, model.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, model.trainable_variables))

        return {"loss": loss}

上記の `train_step` メソッドの実装は、[Keras の `train_step` 規約](https://www.tensorflow.org/guide/keras/customizing_what_happens_in_fit) に従っています。これはオプションですが、トレーニング ステップの動作を変更しながら、Keras の `Model.compile` および `Model.fit` メソッドを引き続き使用することができます。

In [ ]:
model = CustomTraining(
    vocab_size=len(ids_from_chars.get_vocabulary()),
    embedding_dim=embedding_dim,
    rnn_units=rnn_units,
)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
)

In [ ]:
model.fit(dataset, epochs=1)

さらに詳細な制御が必要な場合は、完全に独自のカスタム トレーニング ループを記述することもできます。

In [ ]:
EPOCHS = 10

mean = tf.metrics.Mean()

for epoch in range(EPOCHS):
    start = time.time()

    mean.reset_states()
    for batch_n, (inp, target) in enumerate(dataset):
        logs = model.train_step([inp, target])
        mean.update_state(logs["loss"])

        if batch_n % 50 == 0:
            template = (
                f"Epoch {epoch+1} Batch {batch_n} Loss {logs['loss']:.4f}"
            )
            print(template)

    # 5 エポックごとにモデルを保存（チェックポイント作成）
    if (epoch + 1) % 5 == 0:
        model.save_weights(checkpoint_prefix.format(epoch=epoch))

    print()
    print(f"Epoch {epoch+1} Loss: {mean.result().numpy():.4f}")
    print(f"Time taken for 1 epoch {time.time() - start:.2f} sec")
    print("_" * 80)

model.save_weights(checkpoint_prefix.format(epoch=epoch))